Wait, I think I actually need a lexer before the parser.

In [121]:
from enum import Enum

class LexerTokenEnum(Enum):
    """Enum of lexer token options"""
    ACTIVITY = 1
    COMMENT = 2
    TIME = 3
    SENTENCE = 4
    END_OF_INPUT = 5

In [122]:
class LexerNoMatchError(Exception):
    """Error to handle a lexer token not matching"""
    pass

In [123]:
class Lexeme:
    """Class to hold an extracted lexer element"""
    def __init__(self, identifier: LexerTokenEnum, text: str):
        self.identifier = identifier
        self.text = text

In [ ]:
class LexerToken:
    """Class to define tokens"""
    def __init__(self, search_term: str, lexer_token: LexerTokenEnum):
        self.search_term = search_term
        self.lexer_token = lexer_token

    def _checkInd(self, lexer_char: str, check_pos: int) -> bool:
        """Method to check individual characters of the text and the token"""
        if lexer_char == self.search_term[check_pos]:
            return True
        return False

    def checkChar(self, lexer_str: str) -> Lexeme:
        """Method to check if the text matches a token"""        
        if len(lexer_str) < len(self.search_term):
            raise LexerNoMatchError("LexerLogic.checkChar() lexer_str smaller than self.search_term")
        
        if len(lexer_str) > 0 and len(lexer_str) >= len(self.search_term):
            for char_pos in range(len(self.search_term)):
                if not self._checkInd(lexer_str[char_pos], char_pos):
                    raise LexerNoMatchError("LexerLogic.checkChar() no match")
            return Lexeme(self.lexer_token, self.search_term)
        raise LexerNoMatchError("LexerLogic.checkChar() no match")

In [125]:
class LexerTokens:
    """Class to hold tokens for the lexer"""
    def __init__(self):
        self.tokens: list[LexerToken] = []
    
    def addToken(self, token: LexerToken):
        """Method to add tokens to the container"""
        if not isinstance(token, LexerToken):
            raise TypeError("LexerTokens.addToken() token is not a LexerToken")
        self.tokens.append(token)

In [126]:
at_at_mode = LexerToken(search_term = "@@", lexer_token = LexerTokenEnum.TIME)
at_less_mode = LexerToken(search_term = "@<", lexer_token = LexerTokenEnum.COMMENT)
greater_at_mode = LexerToken(search_term = ">@", lexer_token = LexerTokenEnum.ACTIVITY)

lexer_tokens = LexerTokens()
lexer_tokens.addToken(at_at_mode)
lexer_tokens.addToken(at_less_mode)
lexer_tokens.addToken(greater_at_mode)

In [ ]:
class Lexer:
    """Class for the main lexer"""
    def __init__(self, lexer_tokens: LexerTokens, src_txt: str):
        self.lexer_tokens: LexerTokens = lexer_tokens
        self.lexed: list[Lexeme] = []
        self.buffer: str = ""
        self.stack = src_txt

    def _itrChar(self):
        """Method to move 1 character from the stack to the buffer"""
        if len(self.stack) >= 1:
            self.buffer = "".join((self.buffer, self.stack[0]))
            self.stack = self.stack[1:]
        else:
            raise IndexError("Lexer._itrChar() there are no more characters in self.stack")
        
    def _addBuffer(self):
        """Method to add the buffer as a LexerTokenEnum.SENTENCE lexeme to self.lexed"""
        if len(self.buffer) != 0 and len(self.buffer.strip()) > 0:
                self.lexed.append(Lexeme(LexerTokenEnum.SENTENCE, self.buffer.strip()))
                self.buffer = ""

    def _addLexeme(self, lexeme: Lexeme):
        """Method to add the matched Lexeme to to self.lexed"""
        if isinstance(lexeme, Lexeme):
            self.lexed.append(lexeme)
            self.stack = self.stack[len(lexeme.text):]
        else:
            raise TypeError("Lexer._addLexeme() lexeme is not a Lexeme")

    def _addMatch(self, lexeme: Lexeme):
        """Method to add lexemes to self.lexed after a token is matched"""
        self._addBuffer()
        self._addLexeme(lexeme)
        
    def _checkChar(self) -> Lexeme:
        """Method to check if the current stack position matches a token"""
        for token in self.lexer_tokens.tokens:
            try:
                return token.checkChar(self.stack)
            except:
                pass
        raise LexerNoMatchError("Lexer._checkChar() no token match")
            
    def searchLexer(self):
        """Method to search the Lexer"""
        while True:
            try:
                self._addMatch(self._checkChar())
            except:
                self._itrChar()
            if len(self.stack) == 0:
                if len(self.buffer) >= 1:
                    self._addBuffer()
                self.lexed.append(Lexeme(LexerTokenEnum.END_OF_INPUT, ""))
                break

In [128]:
#test_str = "2026-04-24 07:29 @@ Bathroom >@ Reading Berserk @< Cool Chapter >@ @< Chapter 176"
#test_str = "2026-04-24 07:29 @@ Bathroom >@ Reading Berserk @< Cool Chapter >@@< Chapter 176"
test_str = "Cool Chapter >@@< Chapter 176"
#test_str = "@< Chapter 176"
test_lexer = Lexer(lexer_tokens, test_str)
test_lexer.searchLexer()
print(test_lexer.stack)
print(test_lexer.buffer)
print(test_lexer.lexed)
print(len(test_lexer.lexed))
for val in test_lexer.lexed:
    print(val.identifier)
    print(val.text)
    print("\n")

Cool Chapter >@@< Chapter 176

C
@
Booo
----
C
@
Booo
----
C
>
Booo
----
ool Chapter >@@< Chapter 176
C
o
@
Booo
----
o
@
Booo
----
o
>
Booo
----
ol Chapter >@@< Chapter 176
Co
o
@
Booo
----
o
@
Booo
----
o
>
Booo
----
l Chapter >@@< Chapter 176
Coo
l
@
Booo
----
l
@
Booo
----
l
>
Booo
----
 Chapter >@@< Chapter 176
Cool
 
@
Booo
----
 
@
Booo
----
 
>
Booo
----
Chapter >@@< Chapter 176
Cool 
C
@
Booo
----
C
@
Booo
----
C
>
Booo
----
hapter >@@< Chapter 176
Cool C
h
@
Booo
----
h
@
Booo
----
h
>
Booo
----
apter >@@< Chapter 176
Cool Ch
a
@
Booo
----
a
@
Booo
----
a
>
Booo
----
pter >@@< Chapter 176
Cool Cha
p
@
Booo
----
p
@
Booo
----
p
>
Booo
----
ter >@@< Chapter 176
Cool Chap
t
@
Booo
----
t
@
Booo
----
t
>
Booo
----
er >@@< Chapter 176
Cool Chapt
e
@
Booo
----
e
@
Booo
----
e
>
Booo
----
r >@@< Chapter 176
Cool Chapte
r
@
Booo
----
r
@
Booo
----
r
>
Booo
----
 >@@< Chapter 176
Cool Chapter
 
@
Booo
----
 
@
Booo
----
 
>
Booo
----
>@@< Chapter 176
Cool Chapter 
>
@
Booo
----
>
@
Bo